# 6 . EMA Breakout — candle chart (opens in your browser)

A fresh, self-contained copy of the **Simple 1m EMA Breakout** study whose chart shows **native
candlesticks**.

### Why the chart opens in a browser tab instead of inline
The chart is a normal plotly figure with a `go.Candlestick` trace (SVG). Exported to a standalone
HTML it renders candles perfectly in any browser — but *this* notebook front-end's **inline**
plotly renderer fails to draw the candlestick trace (the line/EMA/marker traces render, the
candles don't). Rather than fight the front-end, the chart cell writes a self-contained HTML and
**opens it in a new browser tab**, which renders reliably every time.

> **Run order:** Kernel → Restart & Run All. If you edited library code, restart so the latest
> `quant` loads. Each run of the chart cell opens (or refreshes) `ema_breakout_chart.html`.

## 0 . Setup

In [11]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import webbrowser
from pathlib import Path
import pandas as pd
from quant.data import get_ohlcv
from quant.engine import BacktestConfig
from quant.strategies import EmaCross
from quant.reporting import print_summary
from quant.viz import ResearchChart
pd.set_option('display.max_rows', 200)
print('quant', quant.__version__)

quant 0.1.0


## 1 . Data (1-minute gold)
A short window keeps candles legible; widen `START`/`END` later. The cache lives at `<repo>/data/`
(resolved from the package, not the notebook folder).

In [12]:
# SYMBOL, SOURCE, MARKET = 'PAXGUSDT', 'binance', 'spot'
# START, END = '2026-07-21', '2026-07-22'
# df = get_ohlcv(SYMBOL, '1m', start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
SYMBOL, SOURCE, MARKET = 'XAUUSD', 'dukascopy', 'cfd'
START, END = '2026-07-21', '2026-07-22'
df = get_ohlcv(SYMBOL, '1m', start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} 1m bars  {df.open_time.min()} -> {df.open_time.max()}')

[07:36:08] INFO | quant.data | cache HIT  dukascopy XAUUSD 1m -> pushdown load


1,381 1m bars  2026-07-21 00:00:00+00:00 -> 2026-07-22 00:00:00+00:00


## 1b . Entry-confirmation variant switches
Two independent toggles on top of `entry_mode='full_candle'`:
- `USE_HEIKIN_ASHI`: evaluate the EMA *and* the candle checks on Heikin Ashi values
  (`ha_close`/`ha_low`/`ha_high`/`ha_open`) instead of real OHLC — full HA mode, not just a
  cosmetic candle skin.
- `CONFIRM_N` / `CONFIRM_COLOR`: `CONFIRM_N` sets how many consecutive candles must be fully
  above (long) / below (short) the entry EMA; `CONFIRM_COLOR=True` additionally requires those
  same `CONFIRM_N` candles to be green (long) / red (short). The two are independent — use either
  alone or both together.

In [13]:
USE_HEIKIN_ASHI = True   # True -> EMA + candle checks evaluated on Heikin Ashi values
CONFIRM_N = 1             # consecutive candles required fully above (long) / below (short) the EMA
CONFIRM_COLOR = True     # True -> those CONFIRM_N candles must also be green (long) / red (short)

print(f"entry: {CONFIRM_N} consecutive {'Heikin Ashi ' if USE_HEIKIN_ASHI else ''}candle(s) "
      f"fully above EMA200" + (", all green" if CONFIRM_COLOR else ""))

entry: 1 consecutive Heikin Ashi candle(s) fully above EMA200, all green


## 2 . Strategy + config + backtest
Full candle above EMA-200 to enter (**one entry per crossing** — `one_entry_per_cross=True`
re-arms only after price touches EMA-200 again), full candle below EMA-50 to exit, swing-low
structure stop, long only, risk 1% per trade, Exness spread 0.12.

In [14]:
strat = EmaCross(
    ema_period=250, entry_mode='full_candle', confirm_n=CONFIRM_N,  # full candle above EMA200
    confirm_color=CONFIRM_COLOR,                               # optionally: those N must be green too
    use_heikin_ashi=USE_HEIKIN_ASHI,                            # optionally: evaluate on HA candles
    allow_long=True, allow_short=False,                        # long only
    exit_mode='below_ema', exit_ema=20,                        # exit: full candle below EMA20
    swings_left=5, swings_right=5,                             # 'previous low' = 5-bar swing
    one_entry_per_cross=True,                                  # ONE entry per cross: fire once when
                                                               #   the candle first forms fully above
                                                               #   EMA200, then re-arm only once a
                                                               #   candle FULLY forms below EMA200
                                                               #   (a touch/wick through it does not
                                                               #   re-arm it)
)
cfg = BacktestConfig(
    initial_cash=10_000, spread=0.12, slippage_bps=1.0,
    exit_enabled=True,
    sl_mode='ref_col', sl_ref_long_col='swing_last_low',       # stop = previous swing low
    sl_buffer_pct=0.0, sl_fallback_mode='entry_pct', sl_fallback_value=1.0,
    tp_mode='none',                                            # exit rule handles targets
    sizing_mode='risk_pct_equity', sizing_value=1.0,           # risk 1% of equity per trade
    allow_rule_close=True,
)
res = strat.backtest(df, cfg)
print_summary(res, df=df)

[07:36:13] INFO | quant.engine | backtest done | bars=1,381 trades=10 final_cash=9970.15 return=-0.30% dd=0.55% | 0.2 ms


 PERFORMANCE SUMMARY
  Total return %                      -0.30
  Final cash                       9,970.15
  Trades                                 10
  Win rate %                          30.00
  Profit factor                        0.43
  Expectancy / trade                -2.9854
  Sharpe                             -14.63
  Sortino                             -9.60
  Calmar                            -207.04
  Max drawdown %                       0.55
  Recovery factor                     -0.54
  Avg bars held                          21
  Total fees                           0.00

  Best weekday: Tue (pnl=-29.85, win=30.0%)
  Best session: newyork (pnl=3.03, n=3)


## 3 . Build the chart and open it in a browser tab
Writes a self-contained `ema_breakout_chart.html` (plotly.js embedded) next to this notebook and
opens it in a new browser tab. **Click any legend item to toggle it**; hover a marker for full
trade details (each shows its `trade #`, matching the table in section 5).

**Real / Heikin Ashi candle toggle** (top-right buttons, on the plot itself): both candle
representations are always in the chart — click to switch between them, no re-running needed.
The EMA lines and entry/exit markers reflect whatever `USE_HEIKIN_ASHI` (section 1b) is currently
set to, since that's what the strategy actually evaluated.

Re-running this cell overwrites the file and opens a fresh tab.

In [15]:
def open_chart_in_browser(fig, filename='ema_breakout_chart.html'):
    """Write a standalone HTML for `fig` and open it in a new browser tab.
    Reliable across notebook front-ends whose inline renderer won't draw candlesticks."""
    out = Path(filename).resolve()
    fig.write_html(str(out), include_plotlyjs=True, full_html=True)
    webbrowser.open(out.as_uri())
    print('chart opened in your browser ->', out)
    return out

from quant.indicators import add_heikin_ashi

def _as_candle_source(source_df):
    """Rename ha_* -> open/high/low/close (keeping the real ones as real_*) so ResearchChart can
    plot Heikin Ashi candles directly."""
    return add_heikin_ashi(source_df).rename(
        columns={'open': 'real_open', 'high': 'real_high', 'low': 'real_low', 'close': 'real_close',
                 'ha_open': 'open', 'ha_high': 'high', 'ha_low': 'low', 'ha_close': 'close'})

ha_df = _as_candle_source(df)
# Primary candle source matches whatever the strategy actually evaluated (USE_HEIKIN_ASHI), so the
# EMA lines and entry/exit markers line up with what's drawn by default; the OTHER representation
# is added as a toggle-able alt series (see add_candle_alt below) so you can compare visually
# without re-running anything.
if USE_HEIKIN_ASHI:
    primary_df, primary_name, alt_df, alt_name = ha_df, 'Heikin Ashi', df, 'Real candles'
    chart_title = f'{SYMBOL} 1m - EMA breakout (Heikin Ashi)'
else:
    primary_df, primary_name, alt_df, alt_name = df, 'Real candles', ha_df, 'Heikin Ashi'
    chart_title = f'{SYMBOL} 1m - EMA breakout'

# candle_target=len(df): render every candle at NATIVE 1-minute resolution (no aggregation).
# Without this, ResearchChart aggregates down to candle_target=400 buckets whenever the view has
# more bars than that -- each displayed candle then merges several real 1-minute bars, so its low
# can visually dip below an EMA even though the single 1-minute bar that triggered a signal never
# did (verified: every entry's real 1m low is strictly above ema200 at signal time). That's
# misleading exactly when you're trying to eyeball entry/exit logic against the chart. For a
# bigger multi-day/week window, drop this back down (e.g. 400-1000) and zoom into specific trades
# instead of rendering everything natively at once.
ch = ResearchChart(primary_df, candles=True, initial_bars=len(primary_df),
                   candle_target=len(primary_df), title=chart_title, candle_name=primary_name)
ch.add_ema(250); ch.add_ema(200); ch.add_ema(100); ch.add_ema(50); ch.add_ema(20)
ch.add_candle_alt(alt_df, name=alt_name)   # on-plot toggle button, top-right of the chart
ch.add_trades(res.trades)          # entry/exit markers + dotted stop-loss lines
fig = ch.show()
open_chart_in_browser(fig)

chart opened in your browser -> C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\ema_breakout_chart.html


WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/ema_breakout_chart.html')

## 4 . Sanity check — confirm candles are really in the figure
Prints the trace inventory: TWO `candlestick` traces (real + Heikin Ashi, one hidden until you
click the toggle) with N>0 points each, and **no** `scattergl` (WebGL) traces. This proves the
figure is correct even if a front-end refuses to draw it inline.

In [ ]:
for t in fig.data:
    n = len(t.x) if getattr(t, 'x', None) is not None else 0
    print(f'{t.type:12s} {str(t.name):14s} visible={str(t.visible):5s} {n:>6d} pts')
candlesticks = [t for t in fig.data if t.type == 'candlestick']
assert len(candlesticks) == 2, f'expected 2 candlestick traces (real + HA), got {len(candlesticks)}'
assert all(len(t.x) > 0 for t in candlesticks), 'a candlestick trace has no data!'
assert sum(bool(t.visible) for t in candlesticks) == 1, 'exactly one candle trace should be visible'
assert not any(t.type == 'scattergl' for t in fig.data), 'WebGL trace present (candles may hide)!'
print('\nOK: both candle sources present with data, exactly one visible, all overlays are SVG.')

## 5 . Trade table (with close reason)

In [ ]:
cols = ['side','entry_time','entry_price','stop_price','exit_time','exit_price',
        'pnl','return_pct','bars_held','close_reason']
trades = res.trades[cols].copy()
trades['pnl'] = trades['pnl'].round(2); trades['return_pct'] = trades['return_pct'].round(2)
for c in ['entry_price','exit_price','stop_price']: trades[c] = trades[c].round(3)
print('close-reason counts:'); display(trades['close_reason'].value_counts())
trades

## 6 . Zoom straight to one trade
Focus the view on a chosen trade, then re-open the browser tab. Change `.iloc[-1]` (biggest
winner) to `.iloc[0]` (worst) or index by the trade table above.

In [ ]:
tr = res.trades.sort_values('pnl').iloc[-1]
pad = pd.Timedelta(minutes=90)
x0 = pd.to_datetime(tr['entry_time']) - pad
x1 = pd.to_datetime(tr['exit_time']) + pad
fig.update_xaxes(range=[x0, x1])   # candles re-aggregate to this window automatically
print(f"trade #{tr.name}: {tr['side']} {tr['entry_time']} @ {tr['entry_price']:.3f} -> "
      f"{tr['exit_time']} @ {tr['exit_price']:.3f} | stop {tr['stop_price']:.3f} | "
      f"pnl {tr['pnl']:.2f} | {tr['close_reason']}")
open_chart_in_browser(fig)